# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alizawwaris974/Assignment-1---Flyrank-ML/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Logistic Regression, then Random Forest. Per the skill's own table, this is a "yes/no with an observed label" question (is_declining), so the recommended path is readable-first: Logistic Regression as the interpretable baseline model, Random Forest only if it earns its added complexity over LR. Evaluated at Precision@K (10/20/50) to match the ranking nature of the actual decision ("which pages to review first"), not raw accuracy.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [1]:
import duckdb, getpass
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ').strip()
conn = duckdb.connect()
conn.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

momentum_query = f"""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_prev
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_last,
               AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_mar
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_impressions IS NOT NULL
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT mar.*, feb.imp_prev
    FROM mar LEFT JOIN feb
      ON mar.client_hash_id = feb.client_hash_id AND mar.content_hash_id = feb.content_hash_id
    WHERE mar.imp_last IS NOT NULL AND feb.imp_prev >= 100
"""
df = conn.execute(momentum_query).df()

content_query = f"""
    SELECT content_hash_id, content_type, word_count, content_created_date
    FROM read_parquet('{REL}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
"""
content_df = conn.execute(content_query).df()
df = df.merge(content_df, on="content_hash_id", how="inner")  # inner: drop items missing from dim_content, same as w03

df["is_declining"] = (df["imp_last"] < 0.8 * df["imp_prev"]).astype(int)
df["content_age_days"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(df["content_created_date"])).dt.days
df["word_count"] = df["word_count"].fillna(df["word_count"].median())
df["content_type"] = df["content_type"].fillna("unknown")

# Client-grouped split — same principle as notebooks 01/02: a client never appears in both sides
np.random.seed(42)
clients = df["client_hash_id"].unique()
np.random.shuffle(clients)
test_clients = set(clients[:max(1, int(len(clients) * 0.2))])
test_mask = df["client_hash_id"].isin(test_clients)

train_df, test_df = df[~test_mask].copy(), df[test_mask].copy()
print(f"Train: {len(train_df)} rows, {len(clients) - len(test_clients)} clients")
print(f"Test:  {len(test_df)} rows, {len(test_clients)} clients")
print(f"Decline rate — train: {train_df['is_declining'].mean():.3f}, test: {test_df['is_declining'].mean():.3f}")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train: 53715 rows, 28 clients
Test:  23023 rows, 6 clients
Decline rate — train: 0.188, test: 0.165


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- Baseline, recomputed on the TEST split only (fair comparison) ---
test_df["baseline_score"] = np.where(
    (test_df["imp_prev"] >= 100) & (test_df["avg_position_mar"] > 10),
    test_df["imp_prev"], 0
)
print("Baseline nonzero on test:", (test_df["baseline_score"] > 0).sum(), "of", len(test_df))

# --- Honest feature set (no label-derived or future-window inputs) ---
content_type_dummies_train = pd.get_dummies(train_df["content_type"], prefix="ct", drop_first=True)
content_type_dummies_test = pd.get_dummies(test_df["content_type"], prefix="ct", drop_first=True)
content_type_dummies_test = content_type_dummies_test.reindex(columns=content_type_dummies_train.columns, fill_value=0)

feature_cols = ["avg_position_mar", "imp_prev", "content_age_days", "word_count"]
X_train = pd.concat([train_df[feature_cols].fillna(0).reset_index(drop=True), content_type_dummies_train.reset_index(drop=True)], axis=1)
X_test = pd.concat([test_df[feature_cols].fillna(0).reset_index(drop=True), content_type_dummies_test.reset_index(drop=True)], axis=1)
y_train, y_test = train_df["is_declining"].values, test_df["is_declining"].values

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).fit(X_train, y_train)
lr_score = lr.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42).fit(X_train, y_train)
rf_score = rf.predict_proba(X_test)[:, 1]

results = []
for k in (10, 20, 50):
    results.append({
        "k": k,
        "baseline": precision_at_k(test_df["baseline_score"].values, y_test, k),
        "logistic_regression": precision_at_k(lr_score, y_test, k),
        "random_forest": precision_at_k(rf_score, y_test, k),
    })
comparison = pd.DataFrame(results)
comparison["base_rate"] = y_test.mean()
print(comparison.to_string(index=False))

Baseline nonzero on test: 7931 of 23023
 k  baseline  logistic_regression  random_forest  base_rate
10      0.60                 0.40           1.00   0.165226
20      0.45                 0.45           0.95   0.165226
50      0.36                 0.40           0.72   0.165226


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [3]:
# Sanity-check what the model leans on — permutation importance, not just built-in feature_importances_
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc")
importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)
print(importance_df.head(8).to_string(index=False))

# 3 concrete wrong cases from the top-20 ranked by the RF model
test_df["rf_score"] = rf_score
wrong_top20 = test_df.sort_values("rf_score", ascending=False).head(20)
wrong_top20 = wrong_top20[wrong_top20["is_declining"] == 0]  # model flagged it, but it wasn't actually declining
print(wrong_top20[["content_hash_id", "avg_position_mar", "imp_prev", "content_age_days", "rf_score"]].head(3).to_string(index=False))

           feature  importance
        word_count    0.072466
  avg_position_mar    0.039157
  content_age_days    0.011053
 ct_feedly article    0.005193
          imp_prev    0.002107
ct_keyword article    0.000437
         content_hash_id  avg_position_mar  imp_prev  content_age_days  rf_score
content_0fc553ca66178e8c         10.551714     123.0               254  0.755772


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.